# Tweets Word2Vec Pipeline (based on reference notebook)

This notebook replicates the preprocessing → Word2Vec → average-embedding → classifier pipeline from the reference notebook:
`Spam Ham Using Word2vec,AvgWord2vec.ipynb`.

**Notes**
- Python runtime: 3.12.3
- Reference Word2Vec usage found in the notebook: `Word2Vec(words)`
- Classifier detected: `LogisticRegression` — using the same style/parameters where possible.
- Input dataset: `/mnt/data/Tweets.csv`
- Using column **`negative_reason`** as text input, and **`airline_sentiment`** as the target.


In [1]:
# Standard imports
import pandas as pd
import numpy as np
import re
import string
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/BTECH_7TH_SEM/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/BTECH_7TH_SEM/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Load the Tweets dataset
df = pd.read_csv(r"Tweets.csv")
print("Loaded dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
# Show sample
df.head()

Loaded dataset shape: (14640, 15)
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [4]:
# Inspect `negativereason` column
print("negativereason present count:", df['negativereason'].notna().sum())
print(df['negativereason'].value_counts(dropna=False).head())

negativereason present count: 9178
negativereason
NaN                       5462
Customer Service Issue    2910
Late Flight               1665
Can't Tell                1190
Cancelled Flight           847
Name: count, dtype: int64


In [5]:
# Preprocessing function (clean, lowercase, remove URLs, punctuation, stopwords)
url_re = re.compile(r'https?://\S+|www\.\S+')
mention_re = re.compile(r'@\w+')
hashtag_re = re.compile(r'#\w+')

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = url_re.sub('', text)
    text = mention_re.sub('', text)
    text = hashtag_re.sub('', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in STOPWORDS]
    return tokens

# Apply cleaning
df['tokens'] = df['negativereason'].fillna("").apply(clean_text)
df['tokens'].head()

0               []
1               []
2               []
3    [bad, flight]
4     [cant, tell]
Name: tokens, dtype: object

In [6]:
# Train Word2Vec on tokens
sentences = df['tokens'].tolist()
# Using Word2Vec parameters observed in reference (or defaults)
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4, epochs=10)
print("Word2Vec vocab size:", len(model.wv))

Word2Vec vocab size: 17


In [7]:
# Function to compute average embeddings for each token list
import numpy as np

def avg_word2vec(tokens, model, vector_size=100):
    if not tokens:
        return np.zeros(vector_size)
    vecs = []
    for t in tokens:
        if t in model.wv:
            vecs.append(model.wv[t])
    if not vecs:
        return np.zeros(vector_size)
    return np.mean(vecs, axis=0)

vector_size = model.vector_size
X = np.vstack(df['tokens'].apply(lambda x: avg_word2vec(x, model, vector_size)).values)
print("X shape:", X.shape)

X shape: (14640, 100)


In [8]:
# Prepare target labels
y = df['airline_sentiment'].fillna('neutral')
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Label classes:", le.classes_)
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
print("Train/Test shapes:", X_train.shape, X_test.shape)

Label classes: ['negative' 'neutral' 'positive']
Train/Test shapes: (11712, 100) (2928, 100)


In [9]:
# Train classifier (LogisticRegression)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.825136612021858

Classification report:
               precision    recall  f1-score   support

    negative       1.00      0.98      0.99      1835
     neutral       0.55      1.00      0.71       620
    positive       0.00      0.00      0.00       473

    accuracy                           0.83      2928
   macro avg       0.52      0.66      0.57      2928
weighted avg       0.74      0.83      0.77      2928


Confusion matrix:
 [[1796   39    0]
 [   0  620    0]
 [   0  473    0]]


/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/NLP/nlp-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/NLP/nlp-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/NLP/nlp-venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

In [ ]:
# Optional: save Word2Vec model and notebook outputs
model.save('/mnt/data/word2vec_tweets_negative_reason.model')
print("Saved Word2Vec model to /mnt/data/word2vec_tweets_negative_reason.model")

## Summary

This notebook:
- Trains Word2Vec on the `negative_reason` column.
- Computes average Word2Vec embeddings per sample.
- Trains a Logistic Regression classifier on the averaged embeddings to predict `airline_sentiment`.

You can run all cells in order. If you want me to:
- Tune hyperparameters (Word2Vec vector_size, window, epochs) or classifier hyperparameters, I can add GridSearchCV.
- Include `airline_sentiment_confidence` as a sample weight during training, say so and I'll include it.

**Notebook saved to:** `/mnt/data/Tweets_Word2Vec_Pipeline.ipynb`